![Lista 2](../.github/lista2_light.png)

**Aluno:** Eduardo Maciel Alexandre
\
**Nome da base:** Telco-Customer-Churn

## Orientações

- Escolha apenas uma das bases disponíveis e resolva todas as 10 questões usando essa mesma base.
- Desenvolva toda a atividade em Python, no formato de entrega do Google Colab.
- Organize o notebook por questão, com códigos executáveis, saídas geradas e comentários objetivos.
- Não troque de base ao longo da atividade.
- Não faça tratamento manual linha por linha.
- Sempre que necessário, sustente suas decisões com tabelas, métricas, gráficos e resultados do código.

## Importação das bibliotecas

In [10]:
# Bibliotecas principais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modelagem e pré-processamento
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

# Bibliotecas auxiliares para visualização e redes
import networkx as nx

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## Carregamento da base

In [11]:
# Carregamento da base
df = pd.read_csv('../data/Telco-Customer-Churn.csv')

# Visualizacao inicial
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Questao 1 – Definicao do problema analitico

**Enunciado:**  
Faca um codigo em Python para revisar a base escolhida e redefinir o problema analitico da Lista 2 de acordo com o cenario selecionado. A partir da base, determine se a tarefa principal sera mais bem tratada por classificacao probabilistica, regressao, arvore de decisao, analise temporal, analise em rede ou visualizacao analitica. Justifique essa escolha com base nas caracteristicas dos dados e crie uma funcao que produza um resumo tecnico da adequacao da base ao metodo escolhido.

### Raciocinio
A estrategia e condensar o diagnostico no que realmente apoia a tomada de decisao: tipo do alvo, distribuicao do churn, mistura de variaveis e sinais de pre-processamento. Em vez de tentar adivinhar o melhor modelo, a funcao devolve uma recomendacao tecnica curta e coerente com os dados observados, limitando a conclusao ao que a base sustenta.

### Desenvolvimento
Implementar uma funcao reutilizavel que sintetize a base e sugira o metodo principal, exibindo tabelas simples para sustentar a decisao.

In [12]:
# Código da Questão 1
def analytical_problem_review(df: pd.DataFrame, target: str) -> dict:
    summary = {}

    # 1) Dimensões
    summary['dimensions'] = {'lines': int(df.shape[0]), 'columns': int(df.shape[1])}

    # 2) Distribuição do alvo
    target_counts = df[target].value_counts(dropna=False).rename('count')
    target_pct = (df[target].value_counts(dropna=False, normalize=True) * 100).round(2).rename('percent')
    target_table = pd.concat([target_counts, target_pct], axis=1).reset_index()
    target_table.columns = [target, 'count', 'percent']
    summary['target_table'] = target_table

    # 3) Perfil de tipos
    type_table = df.dtypes.astype(str).rename('dtype').reset_index()
    type_table.columns = ['column', 'dtype']
    summary['type_table'] = type_table
    summary['n_categorical'] = int(type_table['dtype'].isin(['object', 'category', 'bool']).sum())
    summary['n_numeric'] = int(type_table['dtype'].isin(['int64', 'float64', 'int32', 'float32']).sum())

    # 4) Ausências
    missing_abs = df.isna().sum().rename('missing_abs')
    missing_pct = (df.isna().mean() * 100).round(2).rename('missing_pct')
    missing_table = pd.concat([missing_abs, missing_pct], axis=1).reset_index()
    missing_table.columns = ['column', 'missing_abs', 'missing_pct']
    missing_table = missing_table[missing_table['missing_abs'] > 0].sort_values(
        by=['missing_abs', 'column'], ascending=[False, True]
    )
    summary['missing_table'] = missing_table

    # 5) Colunas numéricas lidas como texto (sinal para pré-processamento)
    numeric_like = []
    for column in df.select_dtypes(include=['object']).columns:
        series = df[column].astype(str).str.strip()
        numeric_try = pd.to_numeric(series, errors='coerce')
        non_empty = series.ne('').sum()
        convertible = numeric_try.notna().sum()
        ratio = (convertible / non_empty) if non_empty > 0 else 0.0
        if ratio >= 0.8:
            numeric_like.append({
                'column': column,
                'numeric_conversion_rate': round(ratio * 100, 2)
            })
    summary['numeric_like_text'] = pd.DataFrame(numeric_like)

    # 6) Recomendação de método
    target_unique = df[target].dropna().nunique()
    if target_unique <= 10 and not pd.api.types.is_numeric_dtype(df[target]):
        method = 'Classificação probabilística'
        decision = 'Estimar a probabilidade de churn para apoiar a retenção de clientes.'
        justification = [
            'Alvo categórico com poucas classes, adequado para classificação.',
            'Predominio de variáveis categóricas sugere modelos probabilísticos com bom suporte ao pré-processamento.'
        ]
    elif pd.api.types.is_numeric_dtype(df[target]):
        method = 'Regressão'
        decision = 'Estimativa de valor contínuo para apoiar previsão.'
        justification = [
            'Alvo numérico indica problema de previsão contínua.'
        ]
    else:
        method = 'Árvore de decisão'
        decision = 'Modelo interpretável para regras de decisão.'
        justification = [
            'Estrutura mista de variáveis favorece regras interpretáveis.'
        ]
    summary['recommendation'] = {
        'suggested_method': method,
        'analytical_decision': decision,
        'justifications': justification
    }

    return summary


q1_review = analytical_problem_review(df, target='Churn')

print('Resumo da base:')
print(f"- Linhas: {q1_review['dimensions']['lines']}")
print(f"- Colunas: {q1_review['dimensions']['columns']}")
print(f"- Variáveis categóricas: {q1_review['n_categorical']}")
print(f"- Variáveis numéricas: {q1_review['n_numeric']}")

print('\nDistribuição do alvo:')
display(q1_review['target_table'])

print('Ausências detectadas (se houver):')
if q1_review['missing_table'].empty:
    print('Sem valores ausentes identificados.')
else:
    display(q1_review['missing_table'])

print('Colunas possivelmente numéricas em texto (se houver):')
if q1_review['numeric_like_text'].empty:
    print('Nenhuma coluna numérica em texto acima do limiar.')
else:
    display(q1_review['numeric_like_text'])

print('Recomendação de método:')
print(f"- Método sugerido: {q1_review['recommendation']['suggested_method']}")
print(f"- Decisão analítica: {q1_review['recommendation']['analytical_decision']}")
print('- Justificativas:')
for item in q1_review['recommendation']['justifications']:
    print(f"  - {item}")

Resumo da base:
- Linhas: 7043
- Colunas: 21
- Variáveis categóricas: 0
- Variáveis numéricas: 3

Distribuição do alvo:


,Churn,count,percent
0,No,5174,73.46
1,Yes,1869,26.54


Ausências detectadas (se houver):
Sem valores ausentes identificados.
Colunas possivelmente numéricas em texto (se houver):


,column,numeric_conversion_rate
0,TotalCharges,100.0


Recomendação de método:
- Método sugerido: Classificação probabilística
- Decisão analítica: Estimar a probabilidade de churn para apoiar a retenção de clientes.
- Justificativas:
  - Alvo categórico com poucas classes, adequado para classificação.
  - Predominio de variáveis categóricas sugere modelos probabilísticos com bom suporte ao pré-processamento.


### Conclusão da Questão 1
Os resultados mostram que a base tem 7.043 registros e que o churn é um alvo categórico com ocorrência de 26,54% na classe Yes, o que caracteriza um problema de classificação binária com desbalanceamento moderado. Essa taxa base é suficiente para orientar a decisão, mas não autoriza conclusões mais fortes do que os dados permitem.  

Não foram identificadas ausências explícitas pelo teste automático, mas a coluna TotalCharges aparece lida como texto e precisa ser tratada antes da modelagem.  

Assim, a leitura mais útil para a decisão não é afirmar se um cliente individual vai cancelar, e sim estimar a propensão ao churn para priorizar ações de retenção. O valor gerado aqui está em transformar a distribuição da base em critério objetivo de priorização, mantendo a conclusão coerente com os resultados exibidos nas tabelas e no diagnóstico estrutural.

## Questão 2 – Naive Bayes

**Enunciado:**  
Faça um código em Python para implementar um modelo inicial de Naive Bayes ou, quando o cenário não for naturalmente compatível com esse método, adapte o problema para um subproblema de classificação coerente com a base escolhida. Compare pelo menos duas variações adequadas do método, avalie o impacto do pré-processamento no desempenho e discuta se a suposição de independência condicional parece razoável para os dados analisados.

### Raciocínio
Escreva aqui, de forma objetiva, a estratégia adotada para responder à questão.

### Desenvolvimento
Implemente abaixo o código da questão.

In [9]:
# Código da Questão 2

### Conclusão da Questão 2
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 3 – Regressão

**Enunciado:**  
Faça um código em Python para construir um problema de Regressão a partir da base escolhida, seja utilizando o alvo original quando ele for numérico, seja definindo uma variável quantitativa derivada coerente com o cenário. Implemente pelo menos dois modelos de regressão, compare desempenho com métricas adequadas e analise os resíduos, discutindo se há sinais de não linearidade, heterocedasticidade ou influência excessiva de outliers.

### Raciocínio
Escreva aqui, de forma objetiva, a estratégia adotada para responder à questão.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 3

### Conclusão da Questão 3
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 4 – Árvore de Decisão

**Enunciado:**  
Faça um código em Python para treinar e avaliar um modelo de Árvore de Decisão adequado ao problema da base escolhida. Controle profundidade, critérios de divisão e tamanho mínimo de amostras por nó, registrando os resultados em tabela. Depois, interprete a árvore gerada, identifique as variáveis mais relevantes nas divisões e discuta se a interpretabilidade do modelo compensa eventuais perdas de desempenho em comparação com os métodos testados anteriormente.

### Raciocínio
Escreva aqui, de forma objetiva, a estratégia adotada para responder à questão.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 4

### Conclusão da Questão 4
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 5 – Comparação entre modelos

**Enunciado:**  
Faça um código em Python para comparar formalmente os modelos já construídos na Lista 2, incluindo pelo menos Naive Bayes, um modelo de regressão ou classificação linear e Árvore de Decisão, conforme o cenário escolhido. Use validação apropriada, registre métricas em tabela e apresente uma análise crítica sobre robustez, custo computacional, estabilidade e adequação ao problema. Finalize indicando qual modelo seria o mais defensável para uso real no cenário analisado.

### Raciocínio
Escreva aqui, de forma objetiva, a estratégia adotada para responder à questão.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 5

### Conclusão da Questão 5
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 6 – Séries Temporais

**Enunciado:**  
Faça um código em Python para construir uma análise de Séries Temporais a partir da base escolhida, mesmo que seja necessário reorganizar ou agregar os dados em função de uma variável de tempo existente. Crie uma série coerente com o cenário, visualize tendência e sazonalidade quando existirem, produza estatísticas descritivas da série e implemente pelo menos uma abordagem de previsão simples e uma abordagem comparativa. Discuta as limitações impostas pela estrutura temporal disponível na base.

### Raciocínio
Escreva aqui, de forma objetiva, a estratégia adotada para responder à questão.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 6

### Conclusão da Questão 6
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 7 – Análise de Redes

**Enunciado:**  
Faça um código em Python para construir uma representação de Rede a partir da base escolhida ou de atributos derivados dela. Defina nós e arestas de forma coerente com o cenário, gere o grafo em Python e calcule medidas como grau, centralidade ou comunidades, quando fizer sentido. Depois, interprete os resultados e discuta se a análise de redes realmente acrescenta valor ao problema ou se sua aplicação no caso escolhido é metodologicamente fraca.

### Raciocínio
Escreva aqui, de forma objetiva, a estratégia adotada para responder à questão.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 7

### Conclusão da Questão 7
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 8 – Meta-aprendizagem

**Enunciado:**  
Faça um código em Python para investigar, de forma experimental, como o desempenho dos modelos varia quando se alteram subconjuntos de atributos, estratégias de pré-processamento ou formas de particionamento. Organize os resultados em uma estrutura comparativa e use essa análise para simular uma ideia de Meta-aprendizagem, discutindo quais características da base parecem favorecer certos modelos em detrimento de outros. Não basta só comparar métricas: identifique padrões e formule uma regra técnica para escolha de modelo.

### Raciocínio
Escreva aqui, de forma objetiva, a estratégia adotada para responder à questão.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 8

### Conclusão da Questão 8
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 9 – Visualização de Dados

**Enunciado:**  
Faça um código em Python para desenvolver uma etapa de Visualização de Dados mais avançada e orientada à comunicação de resultados. Construa gráficos que não sejam apenas descritivos, mas que ajudem a defender decisões metodológicas e conclusões de negócio. Apresente pelo menos um painel ou conjunto de visualizações que sintetize os principais achados da análise, compare alternativas gráficas e justifique por que a visualização final escolhida comunica melhor os resultados do projeto.

### Raciocínio
Escreva aqui, de forma objetiva, a estratégia adotada para responder à questão.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 9

### Conclusão da Questão 9
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 10 – Consolidação da solução

**Enunciado:**  
Faça um código em Python para consolidar toda a análise da Lista 2 em um fluxo final reutilizável. Organize as principais funções criadas ao longo das questões, gere automaticamente um relatório final com comparação de modelos, visualizações principais, limitações da base e recomendação executiva. Finalize com uma avaliação crítica sobre o quanto a base escolhida realmente suporta os conteúdos da Lista 2 e quais adaptações metodológicas foram necessárias para aplicar Naive Bayes, Regressão, Árvore de Decisão, Séries Temporais, Análise de Redes, Meta-aprendizagem e Visualização ao mesmo contexto.

### Raciocínio
Escreva aqui, de forma objetiva, a estratégia adotada para responder à questão.

### Desenvolvimento
Implemente abaixo o código da questão.

In [ ]:
# Código da Questão 10

### Conclusão da Questão 10
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Conclusão Final

Apresente uma síntese geral do trabalho, destacando:

- principais decisões metodológicas;
- comparação entre os modelos utilizados;
- limitações da base escolhida;
- avaliação final da adequação da solução ao cenário.

**Bom trabalho!**